# Different ways for Prompt Tempelates

In [ ]:
# Suppress warnings (same as before)
def warn(*args, **kwargs):
    pass

import warnings
warnings.warn = warn
warnings.filterwarnings('ignore')

import os
os.environ['ANONYMIZED_TELEMETRY'] = 'False'

# ---- OpenAI + LangChain imports ----
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage
from langchain_core.prompts import PromptTemplate
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.prompts import MessagesPlaceholder # Import MessagesPlaceholder for including multiple messages in a template

from langchain_core.output_parsers import JsonOutputParser, CommaSeparatedListOutputParser # to convert LLM responses into structured data
from langchain_core.pydantic_v1 import BaseModel, Field


In [3]:
# Model configuration
openai_llm = ChatOpenAI(
    model="gpt-4.1-nano",
)

## Prompt templates

In [ ]:
prompt = PromptTemplate.from_template("Tell me one {adjective} joke about {topic}")
input_ = {"adjective": "funny", "topic": "cats"}

prompt.invoke(input_)

StringPromptValue(text='Tell me one funny joke about cats')

In [ ]:
prompt = PromptTemplate(
    template="Tell me one {adjective} joke about {topic}",
    input_variables=["topic"],  # Dynamic variables that will be provided when invoking the chain
    partial_variables={"adjective": "funny"},  # default value for 'adjective'
)

prompt.invoke({"topic": "dogs"}) 
# prompt.invoke({"topic": "dogs", "adjective": "hilarious"}) 


StringPromptValue(text='Tell me one funny joke about dogs')

## Chat prompt templates

In [ ]:
prompt = ChatPromptTemplate.from_messages([
 ("system", "You are a helpful assistant"),
 ("user", "Tell me a joke about {topic}")
])
input_ = {"topic": "cats"}

prompt.invoke(input_)

ChatPromptValue(messages=[SystemMessage(content='You are a helpful assistant'), HumanMessage(content='Tell me a joke about cats')])

## MessagesPlaceholder

In [ ]:
# Create a ChatPromptTemplate with a system message and a placeholder for multiple messages
# MessagesPlaceholder allows for inserting multiple messages at once into the template
prompt = ChatPromptTemplate.from_messages([
("system", "You are a helpful assistant"),
MessagesPlaceholder("msgs")  # This will be replaced with one or more messages
])

# Key should match the MessagesPlaceholder name
input_ = {"msgs": [HumanMessage(content="What is the day after Tuesday?"),
                   AIMessage(content="Wednesday"),
                    HumanMessage(content="What is date after 20th of any month") ]} # Here we're adding multiple messages
prompt.invoke(input_)

ChatPromptValue(messages=[SystemMessage(content='You are a helpful assistant'), HumanMessage(content='What is the day after Tuesday?'), AIMessage(content='Wednesday'), HumanMessage(content='What is date after 20th of any month')])

In [10]:
chain = prompt | openai_llm
response = chain.invoke(input = input_)
print(response)

content='The date after the 20th of any month is the 21st.' response_metadata={'token_usage': {'completion_tokens': 16, 'prompt_tokens': 42, 'total_tokens': 58, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4.1-nano-2025-04-14', 'system_fingerprint': 'fp_7f8eb7d1f9', 'finish_reason': 'stop', 'logprobs': None} id='run-59d747aa-8cca-4398-86b9-cc9f1bb7dbbd-0' usage_metadata={'input_tokens': 42, 'output_tokens': 16, 'total_tokens': 58}


# Output parsers

## JSON parser

In [17]:
# Define your desired data structure.
class Joke(BaseModel):
    setup: str = Field(description="question to set up a joke")
    punchline: str = Field(description="answer to resolve the joke")

In [ ]:
joke_query = "Tell me a joke."

output_parser = JsonOutputParser(pydantic_object=Joke)

format_instructions = output_parser.get_format_instructions() # This generates guidance text that tells the LLM how to format its response

# Passing format instructions here to ensure the LLM returns properly structured data
prompt = PromptTemplate(
    template="Answer the user query.\n{format_instructions}\n{query}\n",
    input_variables=["query"],  # Dynamic variables that will be provided when invoking the chain
    partial_variables={"format_instructions": format_instructions},  # setting default value
)

chain = prompt | openai_llm | output_parser
chain.invoke({"query": joke_query})

{'setup': 'Why did the scarecrow win an award?',
 'punchline': 'Because he was outstanding in his field.'}

## Comma-separated list parser

In [ ]:
# to parse LLM responses into Python lists
output_parser = CommaSeparatedListOutputParser() # Create an instance of the parser that will convert comma-separated text into a Python list

format_instructions = output_parser.get_format_instructions() # These instructions explain to the LLM that it should return items in a comma-separated format

prompt = PromptTemplate(
    template="Answer the user query. {format_instructions}\nList five {subject}.",
    input_variables=["subject"],  # This variable will be provided when the chain is invoked
    partial_variables={"format_instructions": format_instructions},  # setting default value
)

chain = prompt | openai_llm | output_parser
chain.invoke({"subject": "ice cream flavors"}) # Parse the LLM's comma-separated response into a Python list

['vanilla',
 'chocolate',
 'strawberry',
 'mint chocolate chip',
 'cookies and cream']

In [ ]:
# replica of above chain without output parser
chain = prompt | openai_llm
chain.invoke({"subject": "ice cream flavors"}) 

AIMessage(content='vanilla, chocolate, strawberry, mint chocolate chip, cookies and cream', response_metadata={'token_usage': {'completion_tokens': 14, 'prompt_tokens': 46, 'total_tokens': 60, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4.1-nano-2025-04-14', 'system_fingerprint': 'fp_f0bc439dc3', 'finish_reason': 'stop', 'logprobs': None}, id='run-86e10084-73c1-4b89-9821-b5a91cd92a38-0', usage_metadata={'input_tokens': 46, 'output_tokens': 14, 'total_tokens': 60})

## JSON Parser - another example

In [ ]:
class movie_info(BaseModel):
    title: str = Field(description="movie title")
    director: str = Field(description="director name")
    year: int = Field(description="release year")
    genre: str = Field(description="movie genre")

output_parser = JsonOutputParser(pydantic_object=movie_info) # Create JSON parser

# format_instructions = output_parser.get_format_instructions() # Instead of manually writing format instructions like below, we can get them from the parser

# Create the format instructions manually
format_instructions = """RESPONSE FORMAT: Return ONLY a single JSON object—no markdown, no examples, no extra keys.  It must look exactly like:
{
  "title": "movie title",
  "director": "director name",
  "year": 2000,
  "genre": "movie genre"
}

IMPORTANT: Your response must be *only* that JSON.  Do NOT include any illustrative or example JSON."""

# Create prompt template with instructions
prompt_template = PromptTemplate(
    template="""Task: Generate info about the movie "{movie_name}".

    Output Format Instructions: {format_instructions}
""",
    input_variables=["movie_name"],
    partial_variables={"format_instructions": format_instructions},
)

# Create the chain
movie_chain = prompt_template | openai_llm | output_parser

# Test with a movie name
movie_name = "Avengers infinity war"
result = movie_chain.invoke({"movie_name": movie_name})

# Print the structured result
print("Parsed result:")
print(f"Title: {result['title']}")
print(f"Director: {result['director']}")
print(f"Year: {result['year']}")
print(f"Genre: {result['genre']}")

Parsed result:
Title: Avengers: Infinity War
Director: Anthony and Joe Russo
Year: 2018
Genre: Superhero, Action, Adventure
